# 02 · First model: blocking → features → LightGBM → threshold (V1 milestone M1)

A crude end-to-end version of the V1 pipeline:

| Stage | V1 section | What this notebook does |
|---|---|---|
| S2 | §6 | Unicode folding (`unidecode`) plus a small seed abbreviation map |
| S4 | §8 P2 | One inverted-index pass, run per country. Keys are name tokens, address tokens, number+street bigrams, a sorted-name key and name char 4-grams. Pairs are scored by IDF sum; top-K per S1 are kept. |
| S5 | §9 | 20 features: rapidfuzz name/address similarities, number Jaccard, blocking provenance and context |
| S6 | §10.1 | LightGBM, 5-fold GroupKFold by S1, OOF |
| S8/S9 | §12–13 | Exclusivity (each S2/S3 keeps its best S1), one global threshold tuned on OOF, list cap |
| S10 | §14 | Export both TSVs and run the official validator |

**Training sample.** Training uses a **density-preserving sample**: X% of S1, all of their matches, and X% of the unmatched S2/S3. That keeps the candidates-per-S1 density of the full data while fitting in RAM and time.

**Test.** The test set runs at **full scale**.

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # run the notebook from the `version 1` folder
import numpy as np, polars as pl
import er
from er import log
pl.Config.set_fmt_str_lengths(90); pl.Config.set_tbl_width_chars(220); pl.Config.set_tbl_rows(30)

import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

CFG = dict(
    frac=0.10,        # share of train S1 used for training (density-preserving sample)
    top_k=30,         # candidates kept per S1
    cap_frac=2e-4,    # blocking keys with more postings than cap_frac * pool are skipped
    folds=5, seed=0,
    lgb=dict(objective="binary", learning_rate=0.05, num_leaves=63, min_data_in_leaf=100,
             feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, seed=0, verbose=-1,
             num_threads=0),
    rounds=2000, early_stop=50,
)
OUT = Path("output"); er.CACHE.mkdir(exist_ok=True)
CFG

{'frac': 0.1,
 'top_k': 30,
 'cap_frac': 0.0002,
 'folds': 5,
 'seed': 0,
 'lgb': {'objective': 'binary',
  'learning_rate': 0.05,
  'num_leaves': 63,
  'min_data_in_leaf': 100,
  'feature_fraction': 0.8,
  'bagging_fraction': 0.8,
  'bagging_freq': 1,
  'seed': 0,
  'verbose': -1,
  'num_threads': 0},
 'rounds': 2000,
 'early_stop': 50}

## 1 · Training sample (density-preserving)

In [2]:
s1, s23, gt = er.load_split("train")
pairs = er.gt_pairs(gt)
s1s = s1.sample(fraction=CFG["frac"], seed=CFG["seed"])
ps = pairs.join(s1s.select(pl.col("entity_id").alias("s1_id")), on="s1_id")
unmatched = s23.join(pairs.select(pl.col("s23_id").alias("entity_id")), on="entity_id", how="anti") \
               .sample(fraction=CFG["frac"], seed=CFG["seed"])
s23s = pl.concat([s23.join(ps.select(pl.col("s23_id").alias("entity_id")), on="entity_id"), unmatched])
del s1, s23, gt, pairs, unmatched
log(f"sample: {s1s.height:,} S1 | {s23s.height:,} S2/S3 | {ps.height:,} true pairs | "
    f"S2/S3 per S1 = {s23s.height / s1s.height:.2f}")

[17:56:26] sample: 220,682 S1 | 1,030,763 S2/S3 | 762,578 true pairs | S2/S3 per S1 = 4.67


## 2 · Normalization

In [3]:
t = time.time()
s1s, s23s = er.add_norm(s1s), er.add_norm(s23s)
log(f"normalized in {time.time()-t:.0f}s")
print(s23s.select("business_name", "name_n", "business_address", "addr_n").sample(8, seed=2))

norm names:   0%|          | 0/220682 [00:00<?, ?it/s]

norm addrs:   0%|          | 0/220682 [00:00<?, ?it/s]

norm names:   0%|          | 0/1030763 [00:00<?, ?it/s]

norm addrs:   0%|          | 0/1030763 [00:00<?, ?it/s]

[17:56:36] normalized in 11s


shape: (8, 4)
┌─────────────────────────────────────────┬─────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────┐
│ business_name                           ┆ name_n                                  ┆ business_address                                                      ┆ addr_n                                                       │
│ ---                                     ┆ ---                                     ┆ ---                                                                   ┆ ---                                                          │
│ str                                     ┆ str                                     ┆ str                                                                   ┆ str                                                          │
╞═════════════════════════════════════════╪═════════════════════════════════════════╪═════════════════

## 3 · Blocking + blocking metrics (V1 §8.5)

In [4]:
t = time.time()
cand = er.block_all(s1s, s23s, CFG["top_k"], CFG["cap_frac"])
log(f"blocking done in {time.time()-t:.0f}s: {cand.height:,} candidate pairs")

# true pairs as (i1, i23) indices
truth = (ps.join(s1s.select(pl.col("entity_id").alias("s1_id"), pl.col("idx").alias("i1")), on="s1_id")
           .join(s23s.select(pl.col("entity_id").alias("s23_id"), pl.col("idx").alias("i23")), on="s23_id")
           .select("i1", "i23"))
hit = truth.join(cand.select("i1", "i23"), on=["i1", "i23"], how="semi")

def truth_long(df):  # (i1, i23) -> (s1_id, s23_id)
    return df.select(s1_id=s1s["entity_id"][df["i1"].to_numpy()], s23_id=s23s["entity_id"][df["i23"].to_numpy()])

ids1 = s1s["entity_id"].to_list()
T = truth_long(truth)
cty = s1s.select(pl.col("idx").alias("i1"), "cty")
tj = (truth.join(hit.with_columns(f=pl.lit(1)), on=["i1", "i23"], how="left").join(cty, on="i1")
        .with_columns(found=pl.col("f").is_not_null()))
per_c = tj.group_by("cty").agg(pair_completeness=pl.col("found").mean(), n_pairs=pl.len())
full = tj.group_by("i1").agg(all_found=pl.col("found").all())
print(f"pair completeness (recall ceiling): {hit.height / truth.height:.4f}")
print(per_c)
print(f"candidates per S1: mean {cand.height / s1s.height:.1f}, p95 {cand['n_cand'].quantile(0.95)}")
print(f"reduction ratio: {1 - cand.height / (s1s.height * s23s.height):.6f}")
print(f"pairs quality (share of candidates that are true): {hit.height / cand.height:.4f}")
print(f"entity-level ceiling (non-singleton S1 with ALL matches found): {full['all_found'].mean():.4f}")
print(f"ORACLE macro F0.5 on these candidates: {er.macro_f05(truth_long(hit), T, ids1):.4f}")

[17:56:36] us: 132,470 S1 vs 618,816 S2/S3 | posting cap 124 | chunk 32,319


keys S2/S3 us:   0%|          | 0/4 [00:00<?, ?it/s]

keys S1 us:   0%|          | 0/1 [00:00<?, ?it/s]

block us:   0%|          | 0/5 [00:00<?, ?it/s]

[17:56:54] india: 88,212 S1 vs 411,947 S2/S3 | posting cap 83 | chunk 48,549


keys S2/S3 india:   0%|          | 0/3 [00:00<?, ?it/s]

keys S1 india:   0%|          | 0/1 [00:00<?, ?it/s]

block india:   0%|          | 0/2 [00:00<?, ?it/s]

[17:57:07] blocking done in 31s: 6,262,413 candidate pairs


pair completeness (recall ceiling): 0.9366
shape: (2, 3)
┌───────┬───────────────────┬─────────┐
│ cty   ┆ pair_completeness ┆ n_pairs │
│ ---   ┆ ---               ┆ ---     │
│ str   ┆ f64               ┆ u32     │
╞═══════╪═══════════════════╪═════════╡
│ india ┆ 0.924728          ┆ 304760  │
│ us    ┆ 0.944506          ┆ 457818  │
└───────┴───────────────────┴─────────┘
candidates per S1: mean 28.4, p95 30.0
reduction ratio: 0.999972
pairs quality (share of candidates that are true): 0.1141
entity-level ceiling (non-singleton S1 with ALL matches found): 0.8332
ORACLE macro F0.5 on these candidates: 0.9757


## 4 · Features

In [5]:
t = time.time()
X = er.pair_features(cand, s1s, s23s)
X = X.join(truth.with_columns(y=pl.lit(1, pl.Int8)), on=["i1", "i23"], how="left").with_columns(pl.col("y").fill_null(0))
log(f"features: {X.shape} in {time.time()-t:.0f}s | positive rate {X['y'].mean():.3f}")
print(X.select(er.MODEL_FEATS).describe())

features:   0%|          | 0/63 [00:00<?, ?it/s]

[17:57:44] features: (6262413, 23) in 36s | positive rate 0.114


shape: (9, 21)
┌────────────┬────────────┬────────────┬────────────┬───┬────────────┬────────────┬──────────────┬────────────┐
│ statistic  ┆ blk_score  ┆ blk_nkeys  ┆ blk_rank   ┆ … ┆ len_a      ┆ len_b      ┆ addr_empty_b ┆ src3       │
│ ---        ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---        ┆ ---          ┆ ---        │
│ str        ┆ f64        ┆ f64        ┆ f64        ┆   ┆ f64        ┆ f64        ┆ f64          ┆ f64        │
╞════════════╪════════════╪════════════╪════════════╪═══╪════════════╪════════════╪══════════════╪════════════╡
│ count      ┆ 6.262413e6 ┆ 6.262413e6 ┆ 6.262413e6 ┆ … ┆ 6.262413e6 ┆ 6.262413e6 ┆ 6.262413e6   ┆ 6.262413e6 │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0        ┆ … ┆ 0.0        ┆ 0.0        ┆ 0.0          ┆ 0.0        │
│ mean       ┆ 19.079809  ┆ 1.923794   ┆ 15.227959  ┆ … ┆ 21.763104  ┆ 23.798399  ┆ 0.023934     ┆ 0.512278   │
│ std        ┆ 17.807764  ┆ 1.729644   ┆ 8.65838    ┆ … ┆ 6.607027   ┆ 8.560186   ┆ 0.152

## 5 · LightGBM with 5-fold GroupKFold by S1 (OOF)
The progress bar tracks folds. LightGBM prints the train/valid log-loss every 100 rounds.

In [6]:
Xn, y, groups = X.select(er.MODEL_FEATS).to_numpy().astype(np.float32), X["y"].to_numpy(), X["i1"].to_numpy()
oof, best_iters = np.zeros(len(y)), []
for fold, (tr, va) in enumerate(tqdm(list(GroupKFold(CFG["folds"]).split(Xn, y, groups)), desc="CV folds")):
    log(f"fold {fold}: train {len(tr):,} | valid {len(va):,}")
    dtr = lgb.Dataset(Xn[tr], y[tr], feature_name=er.MODEL_FEATS)
    dva = lgb.Dataset(Xn[va], y[va], reference=dtr)
    m = lgb.train(CFG["lgb"], dtr, CFG["rounds"], valid_sets=[dtr, dva], valid_names=["train", "valid"],
                  callbacks=[lgb.log_evaluation(100), lgb.early_stopping(CFG["early_stop"], verbose=True)])
    oof[va] = m.predict(Xn[va], num_iteration=m.best_iteration)
    best_iters.append(m.best_iteration)
    log(f"fold {fold}: best_iter {m.best_iteration} | valid logloss {m.best_score['valid']['binary_logloss']:.5f}")
from sklearn.metrics import roc_auc_score, log_loss
log(f"OOF AUC {roc_auc_score(y, oof):.5f} | OOF logloss {log_loss(y, oof):.5f} | best iters {best_iters}")

CV folds:   0%|          | 0/5 [00:00<?, ?it/s]

[17:57:45] fold 0: train 5,009,930 | valid 1,252,483


Training until validation scores don't improve for 50 rounds


[100]	train's binary_logloss: 0.0126977	valid's binary_logloss: 0.0131618


[200]	train's binary_logloss: 0.00950818	valid's binary_logloss: 0.0104345


...

[200]	train's binary_logloss: 0.00959371	valid's binary_logloss: 0.0102707


[300]	train's binary_logloss: 0.00866358	valid's binary_logloss: 0.00984004


Early stopping, best iteration is:
[263]	train's binary_logloss: 0.00890881	valid's binary_logloss: 0.00982076


[18:07:01] fold 4: best_iter 263 | valid logloss 0.00982


[18:07:03] OOF AUC 0.99985 | OOF logloss 0.00940 | best iters [447, 474, 1973, 562, 263]


## 6 · Decision layer: exclusivity + global threshold (tuned on OOF macro F0.5)

In [7]:
scored = X.select("i1", "i23").with_columns(p=pl.Series(oof))
print(f"all-empty baseline: {er.macro_f05(T.head(0), T, ids1):.4f}")
res = []
for thr in tqdm(np.arange(0.30, 0.95, 0.05).round(2), desc="threshold sweep"):
    for max_k in [11]:
        res.append((thr, max_k, er.macro_f05(truth_long(er.decide(scored, thr, max_k)), T, ids1)))
res = pl.DataFrame(res, schema=["thr", "max_k", "macro_f05"], orient="row").sort("macro_f05", descending=True)
print(res)
THR, MAXK = res.row(0)[:2]
log(f"best threshold {THR} (max_k {MAXK}) -> OOF macro F0.5 = {res.row(0)[2]:.4f}")

all-empty baseline: 0.0557


threshold sweep:   0%|          | 0/13 [00:00<?, ?it/s]

shape: (13, 3)
┌──────┬───────┬───────────┐
│ thr  ┆ max_k ┆ macro_f05 │
│ ---  ┆ ---   ┆ ---       │
│ f64  ┆ i64   ┆ f64       │
╞══════╪═══════╪═══════════╡
│ 0.75 ┆ 11    ┆ 0.959629  │
│ 0.7  ┆ 11    ┆ 0.959605  │
│ 0.65 ┆ 11    ┆ 0.959462  │
│ 0.6  ┆ 11    ┆ 0.959172  │
│ 0.8  ┆ 11    ┆ 0.959146  │
│ 0.55 ┆ 11    ┆ 0.958684  │
│ 0.5  ┆ 11    ┆ 0.958025  │
│ 0.85 ┆ 11    ┆ 0.957865  │
│ 0.45 ┆ 11    ┆ 0.957424  │
│ 0.4  ┆ 11    ┆ 0.956469  │
│ 0.35 ┆ 11    ┆ 0.955565  │
│ 0.9  ┆ 11    ┆ 0.95474   │
│ 0.3  ┆ 11    ┆ 0.954367  │
└──────┴───────┴───────────┘
[18:07:08] best threshold 0.75 (max_k 11) -> OOF macro F0.5 = 0.9596


## 7 · Breakdown: per country, singletons vs. non-singletons, precision/recall

In [8]:
pred = truth_long(er.decide(scored, THR, MAXK))
ns = set(T["s1_id"])
single, nonsingle = [i for i in ids1 if i not in ns], [i for i in ids1 if i in ns]
rows = [("all", er.macro_f05(pred, T, ids1)), ("singletons", er.macro_f05(pred, T, single)),
        ("non-singletons", er.macro_f05(pred, T, nonsingle))]
for c_ in s1s["cty"].unique():
    sub = s1s.filter(pl.col("cty") == c_)["entity_id"].to_list()
    rows.append((f"country={c_}", er.macro_f05(pred, T, sub)))
print(pl.DataFrame(rows, schema=["slice", "macro_f05"], orient="row"))
tp = pred.join(T, on=["s1_id", "s23_id"]).height
print(f"micro precision {tp / max(1, pred.height):.4f} | micro recall {tp / T.height:.4f}")

shape: (5, 2)
┌────────────────┬───────────┐
│ slice          ┆ macro_f05 │
│ ---            ┆ ---       │
│ str            ┆ f64       │
╞════════════════╪═══════════╡
│ all            ┆ 0.959629  │
│ singletons     ┆ 0.963527  │
│ non-singletons ┆ 0.959399  │
│ country=us     ┆ 0.96634   │
│ country=india  ┆ 0.949552  │
└────────────────┴───────────┘
micro precision 0.9918 | micro recall 0.9123


## 8 · Error samples (false positives / false negatives)

In [9]:
show = lambda d: (d.join(s1s.select(pl.col("entity_id").alias("s1_id"), "business_name", "business_address"), on="s1_id")
                   .join(s23s.select(pl.col("entity_id").alias("s23_id"), pl.col("business_name").alias("name_r"),
                                     pl.col("business_address").alias("addr_r")), on="s23_id"))
print("FALSE POSITIVES"); print(show(pred.join(T, on=["s1_id", "s23_id"], how="anti").sample(12, seed=0)).drop("s1_id", "s23_id"))
print("FALSE NEGATIVES (in candidates)")
fn = truth_long(hit).join(pred, on=["s1_id", "s23_id"], how="anti")
print(show(fn.sample(12, seed=0)).drop("s1_id", "s23_id"))

FALSE POSITIVES
shape: (12, 4)
┌─────────────────────────────────┬─────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────┐
│ business_name                   ┆ business_address                                                    ┆ name_r                                      ┆ addr_r                                                             │
│ ---                             ┆ ---                                                                 ┆ ---                                         ┆ ---                                                                │
│ str                             ┆ str                                                                 ┆ str                                         ┆ str                                                                │
╞═════════════════════════════════╪══════════════════════════════════════════════════

## 9 · Final model on the whole training sample

In [10]:
n_rounds = int(np.mean(best_iters) * 1.1)
log(f"training final model for {n_rounds} rounds on {len(y):,} pairs")
dall = lgb.Dataset(Xn, y, feature_name=er.MODEL_FEATS)
final = lgb.train(CFG["lgb"], dall, n_rounds, valid_sets=[dall], valid_names=["train"],
                  callbacks=[lgb.log_evaluation(100)])
final.save_model(er.CACHE / "lgb_v1_m1.txt")
imp = pl.DataFrame({"feature": er.MODEL_FEATS, "gain": final.feature_importance("gain")}).sort("gain", descending=True)
print(imp.with_columns(share=(pl.col("gain") / pl.col("gain").sum()).round(4)))
del X, Xn, dall

[18:07:09] training final model for 818 rounds on 6,262,413 pairs


[100]	train's binary_logloss: 0.0127979


[200]	train's binary_logloss: 0.00968545


[300]	train's binary_logloss: 0.00887157


[400]	train's binary_logloss: 0.0083464


[500]	train's binary_logloss: 0.0079021


[600]	train's binary_logloss: 0.0075692


[700]	train's binary_logloss: 0.00725761


[800]	train's binary_logloss: 0.00740117


shape: (20, 3)
┌──────────────┬─────────────┬────────┐
│ feature      ┆ gain        ┆ share  │
│ ---          ┆ ---         ┆ ---    │
│ str          ┆ f64         ┆ f64    │
╞══════════════╪═════════════╪════════╡
│ blk_rank     ┆ 1.6489e8    ┆ 0.2069 │
│ blk_score    ┆ 8.6976e7    ┆ 0.1091 │
│ addr_tset    ┆ 7.6317e7    ┆ 0.0957 │
│ name_jw      ┆ 7.4599e7    ┆ 0.0936 │
│ addr_ratio   ┆ 6.2559e7    ┆ 0.0785 │
│ name_tsort   ┆ 5.4411e7    ┆ 0.0683 │
│ name_tset    ┆ 4.5855e7    ┆ 0.0575 │
│ blk_rev_rank ┆ 4.3244e7    ┆ 0.0543 │
│ name_in_addr ┆ 3.3553e7    ┆ 0.0421 │
│ blk_gap      ┆ 3.0734e7    ┆ 0.0386 │
│ num_jacc     ┆ 2.8922e7    ┆ 0.0363 │
│ addr_jacc    ┆ 2.6830e7    ┆ 0.0337 │
│ len_b        ┆ 2.3431e7    ┆ 0.0294 │
│ name_ratio   ┆ 1.6486e7    ┆ 0.0207 │
│ name_partial ┆ 1.3162e7    ┆ 0.0165 │
│ len_a        ┆ 6.5232e6    ┆ 0.0082 │
│ n_cand       ┆ 4.2302e6    ┆ 0.0053 │
│ src3         ┆ 3.3355e6    ┆ 0.0042 │
│ blk_nkeys    ┆ 1.0566e6    ┆ 0.0013 │
│ addr_empty_b ┆ 9613.913

## 10 · Test inference at full scale → submission files
Blocking runs on the full test pool, per country, with France included. Features and predictions are computed in row chunks so memory stays bounded.

In [11]:
t = time.time()
te1, te23, _ = er.load_split("test")
te1, te23 = er.add_norm(te1), er.add_norm(te23)
log(f"test loaded + normalized: {te1.height:,} S1 | {te23.height:,} S2/S3 | {time.time()-t:.0f}s")
tcand = er.block_all(te1, te23, CFG["top_k"], CFG["cap_frac"])
log(f"test blocking: {tcand.height:,} pairs ({tcand.height / te1.height:.1f} per S1) | {time.time()-t:.0f}s")

norm names:   0%|          | 0/1732544 [00:00<?, ?it/s]

norm addrs:   0%|          | 0/1732544 [00:00<?, ?it/s]

norm names:   0%|          | 0/9969589 [00:00<?, ?it/s]

norm addrs:   0%|          | 0/9969589 [00:00<?, ?it/s]

[18:11:13] test loaded + normalized: 1,732,544 S1 | 9,969,589 S2/S3 | 110s


...

[18:15:12] india: 809,986 S1 vs 4,717,565 S2/S3 | posting cap 944 | chunk 5,000


keys S2/S3 india:   0%|          | 0/24 [00:00<?, ?it/s]

keys S1 india:   0%|          | 0/5 [00:00<?, ?it/s]

block india:   0%|          | 0/162 [00:00<?, ?it/s]

[18:19:17] test blocking: 50,789,999 pairs (29.3 per S1) | 594s


In [12]:
CH = 3_000_000
p = np.empty(tcand.height, dtype=np.float32)
for lo in tqdm(range(0, tcand.height, CH), desc="test scoring (chunks)"):
    f = er.pair_features(tcand.slice(lo, CH), te1, te23)
    p[lo:lo + f.height] = final.predict(f.select(er.MODEL_FEATS).to_numpy().astype(np.float32))
tscored = tcand.select("i1", "i23").with_columns(p=pl.Series(p))
tmatch = er.decide(tscored, THR, MAXK)
log(f"test matches: {tmatch.height:,} | S1 with >=1 match {tmatch['i1'].n_unique() / te1.height:.3f}")

test scoring (chunks):   0%|          | 0/17 [00:00<?, ?it/s]

features:   0%|          | 0/30 [00:00<?, ?it/s]

features:   0%|          | 0/30 [00:00<?, ?it/s]

features:   0%|          | 0/30 [00:00<?, ?it/s]

features:   0%|          | 0/30 [00:00<?, ?it/s]

...

features:   0%|          | 0/30 [00:00<?, ?it/s]

features:   0%|          | 0/30 [00:00<?, ?it/s]

features:   0%|          | 0/30 [00:00<?, ?it/s]

features:   0%|          | 0/28 [00:00<?, ?it/s]

[18:29:00] test matches: 5,614,017 | S1 with >=1 match 0.935


## 11 · Shift monitor (V1 §16): France vs. the training countries
If France's best-score distribution or its predicted-empty rate differs a lot from US/India, the normalization needs fixing before submitting.

In [13]:
best = tscored.group_by("i1").agg(best_p=pl.col("p").max())
mon = (te1.select(pl.col("idx").alias("i1"), "cty").join(best, on="i1", how="left")
          .join(tmatch.group_by("i1").len("n_match"), on="i1", how="left").fill_null(0)
          .group_by("cty").agg(n=pl.len(), best_p_median=pl.col("best_p").median(),
                               best_p_p10=pl.col("best_p").quantile(0.1),
                               predicted_empty=(pl.col("n_match") == 0).mean(),
                               mean_matches=pl.col("n_match").mean()))
print(mon.sort("cty"))

shape: (3, 6)
┌────────┬────────┬───────────────┬────────────┬─────────────────┬──────────────┐
│ cty    ┆ n      ┆ best_p_median ┆ best_p_p10 ┆ predicted_empty ┆ mean_matches │
│ ---    ┆ ---    ┆ ---           ┆ ---        ┆ ---             ┆ ---          │
│ str    ┆ u32    ┆ f32           ┆ f32        ┆ f64             ┆ f64          │
╞════════╪════════╪═══════════════╪════════════╪═════════════════╪══════════════╡
│ france ┆ 259452 ┆ 0.999973      ┆ 0.962836   ┆ 0.084879        ┆ 2.92754      │
│ india  ┆ 809986 ┆ 0.999939      ┆ 0.970768   ┆ 0.068809        ┆ 3.270975     │
│ us     ┆ 663106 ┆ 0.999972      ┆ 0.996792   ┆ 0.052979        ┆ 3.325286     │
└────────┴────────┴───────────────┴────────────┴─────────────────┴──────────────┘


## 12 · Export + official validator

In [14]:
er.export(te1["entity_id"].to_list(), tcand.select("i1", "i23"), tmatch, te23["entity_id"], OUT)
import subprocess
r = subprocess.run([sys.executable, str(er.ROOT / "student_resource/utils/validate_submission.py"),
                    "--matching", str(OUT / "matching_results.tsv"), "--candidate", str(OUT / "candidate_pairs.tsv"),
                    "--test-dir", str(er.DATA / "test")], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
log(f"validator exit code {r.returncode} ({'PASS' if r.returncode == 0 else 'FIX ISSUES'})")
print(pl.read_csv(OUT / "matching_results.tsv", separator="\t", quote_char=None, infer_schema=False, n_rows=5))

ML Challenge 2026 â€” submission validator
  test dir: C:\amazon ml challenge\student_resource\dataset\test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (112887 empty, 1619657 non-empty).
  candidate_pairs.tsv: 1732544 rows (2089 empty, 1730455 non-empty).

PASS â€” no blocking issues found. Safe to submit.
 
[18:29:51] validator exit code 0 (PASS)


shape: (5, 2)
┌───────────────────┬──────────────────────────────────────────────────────────────────────────────┐
│ source1_entity_id ┆ matched_entity_ids                                                           │
│ ---               ┆ ---                                                                          │
│ str               ┆ str                                                                          │
╞═══════════════════╪══════════════════════════════════════════════════════════════════════════════╡
│ S1-714132312      ┆ S2-187020300,S2-435263846,S2-637340732,S3-625880872,S3-867809779             │
│ S1-106407869      ┆ S2-705547832,S3-585937637,S3-613056593                                       │
│ S1-156285671      ┆ S2-172049742,S2-971574924                                                    │
│ S1-689823050      ┆ S2-65478038,S3-506456729                                                     │
│ S1-921369899      ┆ S2-107533354,S2-808920974,S2-995426671,S3-285097097,S3-